In [1]:
import os
import subprocess

LOCAL_DATA_DIR = "data_deteccion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

# Sincronizamos imágenes y etiquetas
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)

# Bajamos también classes.txt y data.yaml, ubicados en la raíz de damage_classification/
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/classes.txt", f"{LOCAL_DATA_DIR}/classes.txt", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/data.yaml", f"{LOCAL_DATA_DIR}/data.yaml", "--quiet"], check=True)

# Conteo simple para confirmar que la descarga trajo la misma cantidad de imágenes y labels
n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imágenes: {n_images} | Labels: {n_labels}")

Imágenes: 6453 | Labels: 6453


In [2]:
import os

# Leemos las clases de classes.txt. El orden importa.
CLASSES_PATH = f"{LOCAL_DATA_DIR}/classes.txt"

with open(CLASSES_PATH) as f:
    CLASS_NAMES = [line.strip() for line in f if line.strip()]

N_CLASSES = len(CLASS_NAMES)
print(f"Clases cargadas desde {CLASSES_PATH}: {N_CLASSES}")
for i, nombre in enumerate(CLASS_NAMES):
    print(f"  {i}: {nombre}")

from collections import Counter
import glob

conteo = Counter()
archivos_con_error = []
ids_inesperados = set()

# Cada archivo .txt de labels/ corresponde a una imagen; cada línea es un daño anotado
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    with open(label_path) as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            partes = line.split()  # formato: class_id x_center y_center width height

            if len(partes) != 5:
                archivos_con_error.append((label_path, i, "cantidad de valores incorrecta", line))
                continue

            try:
                class_id = int(partes[0])
                xc, yc, w, h = map(float, partes[1:5])
            except ValueError:
                archivos_con_error.append((label_path, i, "valores no numéricos", line))
                continue

            if class_id < 0 or class_id >= N_CLASSES:
                ids_inesperados.add(class_id)
                archivos_con_error.append((label_path, i, f"class_id {class_id} fuera de rango", line))
                continue

            # Las coordenadas YOLO están normalizadas (fracción del ancho/alto), deben estar entre 0 y 1. La ubicación se representa en porcentaje, por ejemplo "el daño se encuentra a 50% de la izquierda" para que funcione en imagenes de distintos tamaños
            if not all(0 <= v <= 1 for v in [xc, yc, w, h]):
                archivos_con_error.append((label_path, i, "coordenadas fuera de rango [0,1]", line))

            conteo[CLASS_NAMES[class_id]] += 1

print("\nConteo de daños por clase:")
for clase, cantidad in conteo.most_common():
    print(f"  {clase}: {cantidad}")

print(f"\nTotal de daños anotados: {sum(conteo.values())}")

if ids_inesperados:
    print(f"\nIDs de clase inesperados encontrados (fuera de 0-{N_CLASSES-1}): {ids_inesperados}")
else:
    print("\nTodos los IDs de clase están dentro del rango esperado")

if archivos_con_error:
    print(f"\nSe encontraron {len(archivos_con_error)} líneas con problemas. Primeras 5:")
    for path, linea_num, motivo, contenido in archivos_con_error[:5]:
        print(f"  {path} (línea {linea_num}): {motivo} → '{contenido}'")
else:
    print("No se encontraron errores de formato en las etiquetas")

Clases cargadas desde data_deteccion/classes.txt: 14
  0: Concrete spalling
  1: Corrosion
  2: Crack
  3: Debris
  4: Detachment crack
  5: Diagonal crack
  6: Flexural/rocking crack
  7: Large displacement
  8: Leaching
  9: Non structural damage
  10: Out-of-plane mechanism
  11: Partial/full collapse
  12: Rebar deformation/buckling
  13: Shear crack

Conteo de daños por clase:
  Diagonal crack: 4490
  Concrete spalling: 4467
  Corrosion: 2927
  Shear crack: 1283
  Flexural/rocking crack: 504
  Non structural damage: 408
  Debris: 307
  Detachment crack: 306
  Crack: 201
  Partial/full collapse: 124
  Leaching: 119
  Rebar deformation/buckling: 42
  Out-of-plane mechanism: 26
  Large displacement: 14

Total de daños anotados: 15218

Todos los IDs de clase están dentro del rango esperado
No se encontraron errores de formato en las etiquetas


In [4]:
import numpy as np

GRID_SIZE = 14  # coincide con la resolución de salida del backbone del modelo 1 (224 -> 14 tras 4 MaxPooling)
DEPTH = 1 + 4 + N_CLASSES  # 1 objectness + 4 coords de la caja + N_CLASSES para el tipo de daño

def parse_label_to_grid(label_path, grid_size=GRID_SIZE, n_classes=N_CLASSES):
    target = np.zeros((grid_size, grid_size, 1 + 4 + n_classes), dtype=np.float32)

    if not os.path.exists(label_path):
        return target  # imagen sin ningún daño anotado, target queda todo en cero

    with open(label_path) as f:
        for line in f:
            partes = line.split()
            class_id = int(partes[0])
            xc, yc, w, h = map(float, partes[1:5])

            # En qué celda de la grilla cae el centro de esta caja
            col = min(int(xc * grid_size), grid_size - 1)
            row = min(int(yc * grid_size), grid_size - 1)

            # Si dos daños caen en la misma celda, nos quedamos con el último (limitación simple)
            target[row, col, 0] = 1.0                     # objectness: hay daño en esta celda
            target[row, col, 1:5] = [xc, yc, w, h]         # coordenadas de la caja
            target[row, col, 5 + class_id] = 1.0           # one-hot de la clase

    return target

In [5]:
# Armamos un array con la cantidad de ejemplos de cada clase, en el mismo orden que CLASS_NAMES
conteo_por_indice = np.array([conteo[nombre] for nombre in CLASS_NAMES], dtype=np.float32)

# Peso inversamente proporcional a la frecuencia (menos ejemplos = más peso),
# suavizado con raíz cuadrada para que el desbalance no sea demasiado extremo
CLASS_WEIGHTS = np.sqrt(1.0 / conteo_por_indice)

# Normalizamos para que el promedio de los pesos sea 1 (referencia central)
CLASS_WEIGHTS = CLASS_WEIGHTS / CLASS_WEIGHTS.mean()

print("Pesos por clase:")
for nombre, peso in zip(CLASS_NAMES, CLASS_WEIGHTS):
    print(f"  {nombre}: {peso:.2f}")

Pesos por clase:
  Concrete spalling: 0.18
  Corrosion: 0.22
  Crack: 0.86
  Debris: 0.69
  Detachment crack: 0.69
  Diagonal crack: 0.18
  Flexural/rocking crack: 0.54
  Large displacement: 3.24
  Leaching: 1.11
  Non structural damage: 0.60
  Out-of-plane mechanism: 2.38
  Partial/full collapse: 1.09
  Rebar deformation/buckling: 1.87
  Shear crack: 0.34


In [6]:
import glob

image_files = sorted(glob.glob(f"{LOCAL_DATA_DIR}/images/*.jpg"))

def get_label_path(image_path):
    return image_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

def repeticiones_por_rareza(label_path):
    """Cuántas veces repetir esta imagen en el dataset, según la clase más rara que contenga."""
    if not os.path.exists(label_path):
        return 1
    max_peso = 1.0
    with open(label_path) as f:
        for line in f:
            class_id = int(line.split()[0])
            max_peso = max(max_peso, CLASS_WEIGHTS[class_id])
    return max(1, round(max_peso))

# Armamos la lista final de archivos, repitiendo los que tienen clases raras
image_files_sobremuestreados = []
for img_path in image_files:
    n_rep = repeticiones_por_rareza(get_label_path(img_path))
    image_files_sobremuestreados.extend([img_path] * n_rep)

print(f"Imágenes originales: {len(image_files)}")
print(f"Imágenes tras sobremuestreo: {len(image_files_sobremuestreados)}")

# Split 80/20 sobre la lista ya sobremuestreada
import random
random.seed(42)
random.shuffle(image_files_sobremuestreados)

n_total = len(image_files_sobremuestreados)
n_train = int(n_total * 0.8)
train_paths = image_files_sobremuestreados[:n_train]
val_paths = image_files_sobremuestreados[n_train:]

print(f"Train: {len(train_paths)} | Val: {len(val_paths)}")

Imágenes originales: 6453
Imágenes tras sobremuestreo: 6539
Train: 5231 | Val: 1308


In [7]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def load_sample(image_path):
    image_path = image_path.numpy().decode("utf-8")
    label_path = image_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)

    target = parse_label_to_grid(label_path)
    return img_array, target

def tf_load_sample(image_path):
    img, target = tf.py_function(load_sample, [image_path], [tf.float32, tf.float32])
    img.set_shape([224, 224, 3])
    target.set_shape([GRID_SIZE, GRID_SIZE, 1 + 4 + N_CLASSES])
    return img, target

train_ds = tf.data.Dataset.from_tensor_slices(train_paths).map(
    tf_load_sample, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(val_paths).map(
    tf_load_sample, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Datasets armados")

2026-08-08 19:11:07.899175: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Datasets armados


In [8]:
modelo1 = tf.keras.models.load_model("../models/best_model.keras")

# Buscamos la última capa MaxPooling2D del modelo 1 (fin del backbone puro,
# antes de GlobalAveragePooling, Dense, y también antes del data_augmentation
# que solo aplica durante el entrenamiento del modelo 1)
backbone_output = None
for layer in modelo1.layers:
    if isinstance(layer, tf.keras.layers.MaxPooling2D):
        backbone_output = layer.output

backbone = tf.keras.Model(inputs=modelo1.input, outputs=backbone_output, name="backbone_reutilizado")
backbone.trainable = True

print("Backbone reutilizado, forma de salida:", backbone.output_shape)

Backbone reutilizado, forma de salida: (None, 14, 14, 256)


In [10]:
from tensorflow.keras import layers

x = backbone.output  # (14, 14, 256)

x = layers.Conv2D(256, 3, padding="same", activation="relu", name="det_conv2d_1")(x)
x = layers.Conv2D(128, 3, padding="same", activation="relu", name="det_conv2d_2")(x)

depth = 1 + 4 + N_CLASSES  # objectness + 4 coords caja + clases

output = layers.Conv2D(depth, 1, padding="same", activation="linear", name="det_output")(x)

modelo_deteccion = tf.keras.Model(inputs=backbone.input, outputs=output, name="modelo_deteccion")
modelo_deteccion.summary()

Model: "modelo_deteccion"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ det_conv2d_1 (Conv2D)           │ (None, 14, 14, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ det_conv2d_2 (Conv2D)           │ (None, 14, 14, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ det_output (Conv2D)             │ (None, 14, 14, 19)     │         2,451 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,277,907 (4.87 MB)

 Trainable params: 1,276,947 (4.87 MB)

 Non-trainable params: 960 (3.75 KB)

In [13]:
CLASS_WEIGHTS_TF = tf.constant(CLASS_WEIGHTS, dtype=tf.float32)

def detection_loss(y_true, y_pred, lambda_coord=5.0, lambda_noobj=0.5):
    obj_mask = y_true[..., 0]
    obj_true = y_true[..., 0]
    obj_pred = tf.sigmoid(y_pred[..., 0])
    loss_obj = tf.keras.losses.binary_crossentropy(obj_true, obj_pred)
    loss_obj = obj_mask * loss_obj + lambda_noobj * (1 - obj_mask) * loss_obj

    box_true = y_true[..., 1:5]
    box_pred = y_pred[..., 1:5]
    loss_box = tf.reduce_sum(tf.square(box_true - box_pred), axis=-1)
    loss_box = obj_mask * loss_box

    class_true = y_true[..., 5:]
    class_pred = tf.nn.softmax(y_pred[..., 5:], axis=-1)
    loss_class = tf.keras.losses.categorical_crossentropy(class_true, class_pred)

    # Peso de la clase correcta en cada celda (0 si la celda está vacía, porque class_true es todo ceros ahí)
    peso_clase = tf.reduce_sum(class_true * CLASS_WEIGHTS_TF, axis=-1)
    loss_class = obj_mask * peso_clase * loss_class

    total = tf.reduce_mean(loss_obj) + lambda_coord * tf.reduce_mean(loss_box) + tf.reduce_mean(loss_class)
    return total

In [14]:
modelo_deteccion.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=detection_loss
)

In [15]:
import os
print(os.getcwd())

/home/sagemaker-user/humanoid-damage-cnn/notebooks
